# Snowball v7 — Notebook B：台股模型實驗室與階段 3 組合建構層

**定位（v7 階段 2 & 階段 3）**：
- **輸入**：直接掛載 Kaggle Dataset（`snowball_us_tw_2000_pit_v7.parquet` 或 `snowball_us_tw_2000_pit.parquet`），**不重新爬蟲、不需要 GPU（CPU 數十秒完成）**。
- **範圍**：依階段 1 停止門檻結論（美股 `IC <= 0` 改配市值型 ETF），本實驗室專注於 **台股（TW 990+ 檔）** 獨立訓練與 **階段 3 組合建構層**。
- **嚴格遵守 v7 七大鐵律**：
  1. **最後 52 週保留期凍結**：`HOLDOUT_CUTOFF = "2025-11-20"`，`2025-11-21 ~ 2026-11-20` 絕不用於選特徵、調參或階段 3 評估。
  2. **Purge Gap = `horizon + 1` 週**（`gap_days >= 35` 天）。
  3. **真實台股交易成本**：整股來回 `0.585%`（手續費 `0.1425%×2` + 證交稅 `0.30%`）、零股來回 `1.185%`（含兩邊 `0.30%` 滑價）、下市保守 `-30%`。
  4. **階段 3 五步依序單獨衡量**（分市場純台股 → 產業上限 `≤ 25%` → Beta/波動/產業中性化 → 緩衝帶降換手 `buffer_mult=2.0` → 事前波動度目標 `≤ 15%`）。
  5. **每一次試驗自動寫入 `results/trial_log.csv` 並計算 Bailey & López de Prado (2014) Deflated Sharpe Ratio**。

In [ ]:
# === Cell 1: 環境初始化與載入最新 Snowball v7 引擎 ===
import json
import os
import subprocess
import sys
from pathlib import Path
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/948787878787/snowball.git"
LOCAL_SCRIPTS = Path("skills/snowball/scripts")
KAGGLE_REPO = Path("/tmp/snowball_repo")

if LOCAL_SCRIPTS.exists():
    SCRIPTS_DIR = LOCAL_SCRIPTS.resolve()
elif (Path.cwd().parent / "skills/snowball/scripts").exists():
    SCRIPTS_DIR = (Path.cwd().parent / "skills/snowball/scripts").resolve()
else:
    if not KAGGLE_REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(KAGGLE_REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(KAGGLE_REPO), "pull", "--ff-only"], check=False)
    SCRIPTS_DIR = KAGGLE_REPO / "skills/snowball/scripts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import evaluate as ev
import portfolio_sim as psim
import train_walkforward as twf

HOLDOUT_CUTOFF = "2025-11-20"
OUT_DIR = Path("/kaggle/working/snowball_artifacts_tw_v7") if Path("/kaggle/working").exists() else Path("./snowball_artifacts_tw_v7")
TRIAL_LOG = OUT_DIR / "trial_log.csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Loaded Snowball v7 scripts from: {SCRIPTS_DIR}")
print(f"Strict 52-week Holdout Freeze Cutoff: {HOLDOUT_CUTOFF}")

In [ ]:
# === Cell 2: 自動掛載 Kaggle Dataset 並篩選純台股面板（若未掛載 Dataset 則自動建立台股 PIT 面板） ===
import build_training_panel as btp
import fetch_pit_fundamentals as fpf

def load_or_build_tw_panel() -> tuple[pd.DataFrame, str]:
    candidates = [
        Path("kaggle_dataset/snowball_us_tw_2000_pit_v7.parquet"),
        Path("../kaggle_dataset/snowball_us_tw_2000_pit_v7.parquet"),
        Path("kaggle_dataset/snowball_us_tw_2000_pit.parquet"),
        Path("../kaggle_dataset/snowball_us_tw_2000_pit.parquet"),
        OUT_DIR / "snowball_tw_1000_pit_v7.parquet",
    ]
    kaggle_in = Path("/kaggle/input")
    if kaggle_in.exists():
        candidates = sorted(kaggle_in.rglob("*snowball*pit*.parquet"), reverse=True) + sorted(kaggle_in.rglob("*.parquet"), reverse=True) + candidates
    for p in candidates:
        if p.exists():
            df = pd.read_parquet(p)
            df["date"] = pd.to_datetime(df["date"]).astype("datetime64[ns]")
            tw = df.loc[df["market"] == "TW"].sort_values(["date", "ticker"]).reset_index(drop=True)
            if not tw.empty:
                return tw, str(p.resolve())

    print("未偵測到已掛載的面板 Dataset，自動啟動台股 1,000 檔 PIT 面板建置（Yahoo 價量 + TWSE/TPEx 歷史估值）...")
    panel = btp.build_combined_panel(markets=("TW",), period="8y", horizons=(4, 12))
    panel["date"] = pd.to_datetime(panel["date"]).astype("datetime64[ns]")
    tw_tickers = sorted(panel["ticker"].unique().tolist())
    start_s = (panel["date"].min() - pd.Timedelta(days=60)).strftime("%Y-%m-%d")
    end_s = panel["date"].max().strftime("%Y-%m-%d")
    pit_tw = fpf.fetch_tw_exchange_pit(tw_tickers, cache_dir=Path("/tmp/pit_cache"), start_date=start_s, end_date=end_s)
    if not pit_tw.empty:
        panel = fpf.merge_pit_into_panel(panel, pit_tw, max_staleness_days=200)
    out_p = OUT_DIR / "snowball_tw_1000_pit_v7.parquet"
    panel.to_parquet(out_p, index=False)
    return panel.sort_values(["date", "ticker"]).reset_index(drop=True), str(out_p)

tw_panel, PANEL_SOURCE = load_or_build_tw_panel()
print(f"Panel source : {PANEL_SOURCE}")
print(f"TW rows      : {len(tw_panel):,} | Tickers: {tw_panel['ticker'].nunique()} | Industries: {tw_panel['industry'].nunique()}")
print(f"Date span    : {tw_panel['date'].min().date()} -> {tw_panel['date'].max().date()} (Frozen > {HOLDOUT_CUTOFF})")

In [ ]:
# === Cell 3: 階段 3.1 純台股獨立 Purged Walk-Forward 訓練（嚴格凍結 date <= 2025-11-20） ===
res_tw = twf.run_purged_walkforward(
    panel=tw_panel,
    horizon=4,
    min_train_weeks=104,
    refit_step_weeks=8,
    ridge_alpha=500.0,
    enable_lgbm=True,
    use_gpu=False,
    top_k=20,
    holdout_cutoff=HOLDOUT_CUTOFF,
    tw_odd_lot=False,
)
exported = twf.export_training_artifacts(res_tw, OUT_DIR)
oos_tw = res_tw["oos_predictions"]
assert pd.Timestamp(oos_tw["date"].max()) <= pd.Timestamp(HOLDOUT_CUTOFF), "Holdout freeze violated!"

print("=== 純台股 Walk-Forward 基礎指標（Top-20，整股成本 0.585%，Purge >= 35 天） ===")
print(twf.format_metrics_table(res_tw["metrics"]).round(4).to_string(index=False))

In [ ]:
# === Cell 4: 階段 3 五步組合建構層依序消融實驗與 Deflated Sharpe 檢驗 ===
# 先算同期台股等權基準（TW EW Benchmark）以計算超額 MDD
ew_w = oos_tw.groupby("date")["fwd_ret_1w_open"].mean().dropna()
ew_nav = (1.0 + ew_w).cumprod()
ew_cagr = float(ew_nav.iloc[-1] ** (52.0 / len(ew_w)) - 1.0)
ew_sharpe = float(ew_w.mean() / ew_w.std() * np.sqrt(52.0))
ew_mdd = float((ew_nav / ew_nav.cummax() - 1.0).min())
print(f"TW 等權基準 (2021-10 -> 2025-11, {len(ew_w)} 週): CAGR={ew_cagr:+.2%}, Sharpe={ew_sharpe:.3f}, MDD={ew_mdd:.2%}\n")

STAGE3_STEPS = [
    ("Step3.1_TW_Only",          dict(max_industry_weight=None, neutralize_scores=False, buffer_mult=1.0,  vol_target_annual=None)),
    ("Step3.2_IndCap25%",        dict(max_industry_weight=0.25, neutralize_scores=False, buffer_mult=1.0,  vol_target_annual=None)),
    ("Step3.3_BetaVolIndNeut",   dict(max_industry_weight=0.25, neutralize_scores=True,  buffer_mult=1.0,  vol_target_annual=None)),
    ("Step3.4_BufferTop20Pct",   dict(max_industry_weight=0.25, neutralize_scores=True,  buffer_mult=10.0, vol_target_annual=None)),
    ("Step3.5_VolTarget15%",     dict(max_industry_weight=0.25, neutralize_scores=True,  buffer_mult=10.0, vol_target_annual=0.15)),
]

ladder_rows = []
for model_name, met in res_tw["metrics"].items():
    sc = f"pred_{model_name}"
    for step_name, cfg in STAGE3_STEPS:
        sim_b = psim.simulate_topk_twr(oos_tw, sc, top_k=20, rebalance_every=4, market="TW", tw_odd_lot=False, **cfg)
        sim_o = psim.simulate_topk_twr(oos_tw, sc, top_k=20, rebalance_every=4, market="TW", tw_odd_lot=True, **cfg)
        tid = psim.append_trial_log(
            TRIAL_LOG,
            stage=step_name,
            market="TW",
            model=model_name,
            change=json.dumps(cfg),
            ic=met["TW_Rank_IC"],
            t_nw=met["TW_t_NW"],
            cagr=sim_b["TWR_CAGR"],
            sharpe=sim_b["TWR_Sharpe"],
            mdd=sim_b["TWR_MDD"],
            turnover=sim_b["Avg_Rebalance_Turnover"],
        )
        ladder_rows.append({
            "model": model_name,
            "step": step_name,
            "trial_id": tid,
            "TW_IC": met["TW_Rank_IC"],
            "TW_tNW": met["TW_t_NW"],
            "Board_CAGR": sim_b["TWR_CAGR"],
            "Board_Sharpe": sim_b["TWR_Sharpe"],
            "MDD": sim_b["TWR_MDD"],
            "Excess_MDD_vs_EW": sim_b["TWR_MDD"] - ew_mdd,
            "OddLot_CAGR": sim_o["TWR_CAGR"],
            "OddLot_Sharpe": sim_o["TWR_Sharpe"],
            "Rebal_Turn": sim_b["Avg_Rebalance_Turnover"],
            "Ann_RT": sim_b["Annual_Round_Trips"],
            "Ann_Vol": sim_b["Realized_Annual_Vol"],
            "Avg_Exp": sim_b["Avg_Exposure"],
        })

n_total_trials = len(pd.read_csv(TRIAL_LOG))
n_weeks = int(oos_tw["date"].nunique())
for r in ladder_rows:
    r["DSR"] = ev.deflated_sharpe_ratio(r["Board_Sharpe"], n_trials=n_total_trials, n_obs=n_weeks)

ladder_df = pd.DataFrame(ladder_rows)
ladder_df.to_csv(OUT_DIR / "stage3_tw_ablation_ladder.csv", index=False)
pd.set_option("display.width", 260)
pd.set_option("display.max_columns", 20)
print(ladder_df.round(4).to_string(index=False))

In [ ]:
# === Cell 5: 匯出 P0-1 合規之最新台股選股訊號 CSV（供 broker_exec.py --signals-csv 使用） ===
latest_dt = oos_tw["date"].max()
latest_slice = oos_tw.loc[oos_tw["date"] == latest_dt].copy()
latest_slice = psim.neutralize_score_panel(latest_slice, "pred_stage1_icir_shrinkage", out_col="score")
selected_syms = psim._select_cohort_target(latest_slice, "score", top_k=20, max_industry_weight=0.25)

signals_out = (
    latest_slice.loc[latest_slice["ticker"].isin(selected_syms), ["ticker", "market", "industry", "score", "prob_stage1_icir_shrinkage"]]
    .rename(columns={"prob_stage1_icir_shrinkage": "win_prob"})
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)
sig_path = OUT_DIR / "latest_signals_tw.csv"
signals_out.to_csv(sig_path, index=False)
print(f"Exported {len(signals_out)} industry-capped (<=25%) TW signals for {pd.Timestamp(latest_dt).date()} -> {sig_path}")
print(signals_out.to_string(index=False))